# NeuroGolf 2026 — Baseline Training (Kaggle GPU)

**Model:** TinyCNN (3-layer CNN, ~4K params)  
**Workflow:** Train one model per task on GPU → export to ONNX → verify → zip for submission  

Make sure:
- GPU accelerator is **enabled** (Settings → Accelerator → GPU T4)
- Competition data is **attached** (neurogolf-2026)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
                "onnx", "onnxruntime", "onnx-tool", "-q"], check=True)
print("Packages ready")

In [ ]:
import json
import math
import pathlib
import zipfile

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import onnx
import onnxruntime as ort

DEVICE = torch.device("cpu")  # default
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    gpu_name = torch.cuda.get_device_name(0)
    if major >= 7:
        DEVICE = torch.device("cuda")
        print(f"Device: cuda — {gpu_name} (sm_{major}{minor})")
    else:
        print(f"Device: cpu — {gpu_name} (sm_{major}{minor}) not supported by PyTorch 2.x, training on CPU")
else:
    print("Device: cpu — no GPU detected")

In [ ]:
# Auto-detect data path 
_CANDIDATE_PATHS = [
    "/kaggle/input/neurogolf-2026",
    "/kaggle/input/competitions/neurogolf-2026",
]
DATA_DIR = None
for _p in _CANDIDATE_PATHS:
    if list(pathlib.Path(_p).glob("task*.json")):
        DATA_DIR = pathlib.Path(_p)
        break

if DATA_DIR is None:
    raise RuntimeError(f"Could not find task files. Tried: {_CANDIDATE_PATHS}\n"
                       "Make sure the competition data is attached to this notebook.")

ONNX_DIR = pathlib.Path("/kaggle/working/onnx")
CKPT_DIR = pathlib.Path("/kaggle/working/checkpoints")
ONNX_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

NUM_TASKS = 400
task_files = sorted(DATA_DIR.glob("task*.json"))
print(f"Data directory: {DATA_DIR}")
print(f"Found {len(task_files)} task files")
print(f"Device: {DEVICE}")

## Dataset

In [ ]:
CHANNELS, HEIGHT, WIDTH = 10, 30, 30

def grid_to_tensor(grid):
    """ARC color grid → one-hot numpy array (10, 30, 30). Vectorized."""
    grid_np = np.array(grid, dtype=np.int64)
    h, w = grid_np.shape
    t = np.zeros((CHANNELS, HEIGHT, WIDTH), dtype=np.float32)
    rows, cols = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    t[grid_np, rows, cols] = 1.0
    return t


def load_task_tensors(task_path, splits=("train", "arc-gen")):
    """Load a task JSON and return (inputs, outputs) as stacked tensors."""
    with open(task_path) as f:
        data = json.load(f)
    xs, ys = [], []
    for split in splits:
        for ex in data.get(split, []):
            inp = ex["input"]
            if len(inp) > HEIGHT or len(inp[0]) > WIDTH:
                continue
            xs.append(grid_to_tensor(inp))
            ys.append(grid_to_tensor(ex["output"]))
    if not xs:
        return None, None
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys))


# Pre-load ALL tasks into GPU memory once — eliminates CPU bottleneck during training
print("Pre-loading all tasks into GPU memory...")
all_tasks = {}
for task_num in range(1, NUM_TASKS + 1):
    task_path = DATA_DIR / f"task{task_num:03d}.json"
    if not task_path.exists():
        continue
    xs, ys = load_task_tensors(task_path, splits=("train", "arc-gen"))
    if xs is not None:
        all_tasks[task_num] = (xs.to(DEVICE), ys.to(DEVICE))

print(f"Loaded {len(all_tasks)} tasks into {DEVICE} memory")

## Model — TinyCNN Baseline

In [ ]:
class TinyCNN(nn.Module):
    """~4K params. Input/output: (1, 10, 30, 30) one-hot grids."""
    def __init__(self, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(10, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, 10, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)


m = TinyCNN()
n_params = sum(p.numel() for p in m.parameters())
print(f"TinyCNN params: {n_params:,}")

## Training Loop

In [ ]:
def train_task(task_num, epochs=2000, lr=0.01, hidden=16, batch_size=64):
    if task_num not in all_tasks:
        return False

    xs, ys = all_tasks[task_num]   # already on GPU
    n = xs.shape[0]

    model = TinyCNN(hidden=hidden).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=100, factor=0.5, min_lr=1e-5
    )
    criterion = nn.BCEWithLogitsLoss()

    best_loss = float("inf")
    ckpt_path = CKPT_DIR / f"task{task_num:03d}.pt"

    for epoch in range(1, epochs + 1):
        model.train()
        # Shuffle indices on GPU
        idx = torch.randperm(n, device=DEVICE)
        total_loss = 0.0
        for start in range(0, n, batch_size):
            batch_idx = idx[start:start + batch_size]
            x, y = xs[batch_idx], ys[batch_idx]
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step(total_loss)

        if total_loss < best_loss:
            best_loss = total_loss
            torch.save(model.state_dict(), ckpt_path)

        if total_loss < 1e-6:
            break

    return True

In [ ]:


HIDDEN = 16       
EPOCHS = 2000     
LR = 0.01

trained = []
for task_num in range(1, NUM_TASKS + 1):
    ok = train_task(task_num, epochs=EPOCHS, lr=LR, hidden=HIDDEN)
    if ok:
        trained.append(task_num)
    if task_num % 50 == 0:
        print(f"Progress: {task_num}/{NUM_TASKS} tasks processed, {len(trained)} trained")

print(f"\nDone. Trained {len(trained)} tasks.")

## Export to ONNX

In [ ]:
FILESIZE_LIMIT = 1.44 * 1024 * 1024
_DUMMY = torch.zeros(1, 10, 30, 30)

def export_task(task_num, hidden=HIDDEN):
    ckpt_path = CKPT_DIR / f"task{task_num:03d}.pt"
    if not ckpt_path.exists():
        return False

    onnx_path = ONNX_DIR / f"task{task_num:03d}.onnx"
    model = TinyCNN(hidden=hidden)
    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    model.eval()

    torch.onnx.export(
        model,
        _DUMMY,
        str(onnx_path),
        dynamo=False,       
        opset_version=17,
        input_names=["input"],
        output_names=["output"],
    )

    size = onnx_path.stat().st_size
    return size <= FILESIZE_LIMIT

In [ ]:
exported, over_limit = [], []
for task_num in trained:
    ok = export_task(task_num)
    if ok:
        exported.append(task_num)
    else:
        over_limit.append(task_num)

print(f"Exported: {len(exported)} | Over limit: {len(over_limit)}")
if over_limit:
    print(f"Over-limit tasks: {over_limit}")

## Verify ONNX Models

In [ ]:
def verify_task(task_num):
    onnx_path = ONNX_DIR / f"task{task_num:03d}.onnx"
    task_path = DATA_DIR / f"task{task_num:03d}.json"
    if not onnx_path.exists():
        return None

    with open(task_path) as f:
        data = json.load(f)

    session = ort.InferenceSession(str(onnx_path))

    def check(examples):
        ok, total = 0, 0
        for ex in examples:
            if len(ex["input"]) > 30 or len(ex["input"][0]) > 30:
                continue
            inp = np.zeros((1, 10, 30, 30), dtype=np.float32)
            exp = np.zeros((1, 10, 30, 30), dtype=np.float32)
            for r, row in enumerate(ex["input"]):
                for c, color in enumerate(row):
                    inp[0, color, r, c] = 1.0
            for r, row in enumerate(ex["output"]):
                for c, color in enumerate(row):
                    exp[0, color, r, c] = 1.0
            pred = (session.run(["output"], {"input": inp})[0] > 0).astype(float)
            total += 1
            ok += np.array_equal(pred, exp)
        return ok, total

    arc_ok, arc_total = check(data["train"] + data["test"])
    gen_ok, gen_total = check(data.get("arc-gen", []))
    passed = (arc_ok == arc_total) and (gen_ok == gen_total)
    return passed, arc_ok, arc_total, gen_ok, gen_total

In [ ]:
passed_tasks, failed_tasks = [], []
for task_num in exported:
    result = verify_task(task_num)
    if result is None:
        continue
    passed, arc_ok, arc_total, gen_ok, gen_total = result
    if passed:
        passed_tasks.append(task_num)
    else:
        failed_tasks.append((task_num, arc_ok, arc_total, gen_ok, gen_total))

print(f"Passed: {len(passed_tasks)} / {len(exported)}")
print(f"Failed: {len(failed_tasks)}")
if failed_tasks:
    print("\nFailed tasks (task, arc_ok/total, gen_ok/total):")
    for t, a_ok, a_tot, g_ok, g_tot in failed_tasks[:20]:
        print(f"  task{t:03d}: arc={a_ok}/{a_tot} gen={g_ok}/{g_tot}")

## Package for Submission

In [ ]:
submit_path = pathlib.Path("/kaggle/working/submission.zip")

with zipfile.ZipFile(submit_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for task_num in passed_tasks:
        onnx_path = ONNX_DIR / f"task{task_num:03d}.onnx"
        zf.write(onnx_path, onnx_path.name)

zip_size = submit_path.stat().st_size
print(f"submission.zip: {zip_size:,} bytes ({zip_size/1024/1024:.2f} MB)")
print(f"Contains {len(passed_tasks)} ONNX files")
print(f"\nFile saved at: {submit_path}")
print("Download it from the Kaggle output panel and submit at:")
print("https://www.kaggle.com/competitions/neurogolf-2026/submit")